# 03 — TorchAO CPU INT8 PTQ

Notes:
- `backend='torchao_cpu_ptq'`, `device='cpu'`, `model_precision='int8'`.
- calib on `val` set


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [ ]:
import json
import pandas as pd

from src.config import ExperimentConfig, with_overrides
from src.runner import run_experiment
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)

In [3]:

base = ExperimentConfig(
    backend="torchao_cpu_ptq",
    device="cpu",
    batch_size=1,             # smaller batch for CPU
    model_precision="int8",     # model precision    
    seed=42,
    num_eval_batches = 500,
    cpu_calib_num_batches=1,
)
in_bits_list = [8, 4, 2, 1] 

# total 500 images = 1 * 500
cfgs = [with_overrides(base, input_quant_bits=b) for b in in_bits_list]


In [ ]:
records = []
for cfg in cfgs:
    result_path = cfg.result_json_path()
    if result_path.exists():
        with open(result_path) as f:
            payload = json.load(f)
        print(f"  SKIP {cfg.run_id()} (loaded from disk)")
    else:
        payload, _ = run_experiment(cfg, save_results_flag=True)

    records.append(payload)

for r in records:
    print(r["run_id"], r["status"], r["results"].get("top1_acc"))

In [5]:
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

df_int8_cpu = df[
    (df["cfg.backend"] == "torchao_cpu_ptq")
    & (df["cfg.device"] == "cpu")
].copy()

df_int8_cpu[[
    "run_id",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["run_id"])

,run_id,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
8,resnet18_torchao_cpu_ptq_int8_in1b_cpu_bs1,30.212766,56.170213,11.781395,84.879595,470
9,resnet18_torchao_cpu_ptq_int8_in2b_cpu_bs1,52.127660,78.510638,13.354555,74.880818,470
10,resnet18_torchao_cpu_ptq_int8_in4b_cpu_bs1,83.404255,97.234043,11.755272,85.068213,470
11,resnet18_torchao_cpu_ptq_int8_in8b_cpu_bs1,87.021277,97.872340,11.411533,87.630645,470
